# ****1ER MODELO (Incluyendo histopath_type y lesion_ISUP)****
---
Probablemente sea el más completo porque "lesion_GS" (Gleason score: evaluación de la agresividad del cáncer de próstata) es una variable que da la respuesta de si tiene cáncer o no

DATA LOADING & EDA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
df = pd.read_csv('temp_database1.csv', sep=';')

In [ ]:
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

   patient_id  study_id    mri_date  patient_age   psa  psad  prostate_volume  \
0       11093   1001116  2020-02-06           56   4.9  0.20             24.0   
1       11345   1001368  2018-06-11           67   NaN   NaN             56.0   
2       10415   1000422  2020-06-23           65  15.0  0.27             55.0   
3       10406   1000413  2020-08-15           55   4.6  0.11             42.0   
4       10443   1000451  2019-03-17           64  17.0  0.22             78.0   

  histopath_type lesion_GS lesion_ISUP  case_ISUP case_csPCa  
0     SysBx+MRBx   3+4,3+3         2.1          2        YES  
1           MRBx       3+4           2          2        YES  
2            NaN       NaN         NaN          0         NO  
3     SysBx+MRBx   0+0,0+0           0          0         NO  
4          SysBx       0+0           0          0         NO  
(749, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 749 entries, 0 to 748
Data columns (total 12 columns):
 #   Column         

PREPROCESSING

In [ ]:
#Encoding of the target variable (if a patient has cancer or not: 1 if YES; O if NO)
df['case_csPCa'] = df['case_csPCa'].map({'YES': 1, 'NO': 0})
print(df.head())

   patient_id  study_id    mri_date  patient_age   psa  psad  prostate_volume  \
0       11093   1001116  2020-02-06           56   4.9  0.20             24.0   
1       11345   1001368  2018-06-11           67   NaN   NaN             56.0   
2       10415   1000422  2020-06-23           65  15.0  0.27             55.0   
3       10406   1000413  2020-08-15           55   4.6  0.11             42.0   
4       10443   1000451  2019-03-17           64  17.0  0.22             78.0   

  histopath_type lesion_GS lesion_ISUP  case_ISUP  case_csPCa  
0     SysBx+MRBx   3+4,3+3         2.1          2           1  
1           MRBx       3+4           2          2           1  
2            NaN       NaN         NaN          0           0  
3     SysBx+MRBx   0+0,0+0           0          0           0  
4          SysBx       0+0           0          0           0  


In [ ]:
#Drop columns that are not important (dates, type of analysis...)
cols_to_keep = ['patient_age', 'psa', 'psad', 'prostate_volume',
                'histopath_type', 'lesion_ISUP', 'case_csPCa']
df = df[cols_to_keep]

#Null values before OHE
df['histopath_type'] = df['histopath_type'].fillna('Unknown')
df['lesion_ISUP'] = df['lesion_ISUP'].fillna('Missing')

#OHE
df = pd.get_dummies(df, columns=['histopath_type', 'lesion_ISUP'])

#Adjusting the psa column
df = df[df['psa'] < 1000]

In [ ]:
print(df.describe())

       patient_age         psa        psad  prostate_volume  case_csPCa
count   727.000000  727.000000  524.000000       715.000000  727.000000
mean     65.895461   11.564127    0.216920        66.440448    0.290234
std       6.967067   14.243494    0.394606        36.836201    0.454183
min      37.000000    0.440000    0.010000        10.000000    0.000000
25%      61.500000    5.900000    0.090000        40.000000    0.000000
50%      66.000000    8.200000    0.140000        59.000000    0.000000
75%      71.000000   12.780000    0.220225        82.000000    1.000000
max      86.000000  210.000000    7.000000       279.000000    1.000000


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 727 entries, 0 to 748
Data columns (total 51 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_age                727 non-null    int64  
 1   psa                        727 non-null    float64
 2   psad                       524 non-null    float64
 3   prostate_volume            715 non-null    float64
 4   case_csPCa                 727 non-null    int64  
 5   histopath_type_MRBx        727 non-null    bool   
 6   histopath_type_RP          727 non-null    bool   
 7   histopath_type_SysBx       727 non-null    bool   
 8   histopath_type_SysBx+MRBx  727 non-null    bool   
 9   histopath_type_Unknown     727 non-null    bool   
 10  lesion_ISUP_0              727 non-null    bool   
 11  lesion_ISUP_0,0,0          727 non-null    bool   
 12  lesion_ISUP_0,0,1          727 non-null    bool   
 13  lesion_ISUP_0,2,2          727 non-null    bool   
 14 

In [ ]:
# Mean imputation for filling NAs (median imputation may be also applied if outliers)
df = df.fillna(df.median())

In [ ]:
print(df["case_csPCa"].value_counts())

case_csPCa
0    516
1    211
Name: count, dtype: int64


In [ ]:
#Target variable
X = df.drop(columns = ['case_csPCa'])
y = df['case_csPCa']

#Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [ ]:
#Normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

MLP MODEL

In [ ]:
def create_model(input_dim):
    model = models.Sequential([
        layers.Dense(32, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

model = create_model(X_train.shape[1])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.7050970873786407), np.int64(1): np.float64(1.7189349112426036)}


In [ ]:
#Training
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7371 - loss: 0.6709 - val_accuracy: 0.7949 - val_loss: 0.6573
Epoch 2/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8664 - loss: 0.5004 - val_accuracy: 0.8718 - val_loss: 0.5264
Epoch 3/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9224 - loss: 0.3801 - val_accuracy: 0.8803 - val_loss: 0.4084
Epoch 4/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9526 - loss: 0.2814 - val_accuracy: 0.9060 - val_loss: 0.2999
Epoch 5/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9741 - loss: 0.1800 - val_accuracy: 0.9316 - val_loss: 0.2299
Epoch 6/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9828 - loss: 0.1352 - val_accuracy: 0.9573 - val_loss: 0.1849
Epoch 7/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9828 - loss: 0.1013 - val_accuracy: 0.9658 - val_loss: 0.1539
Epoch 8/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9957 - loss: 0.0633 - val_accuracy: 0.9658 - val_loss

In [ ]:
#Evaluation
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.3).astype(int) #threshold of 0.3
print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       104
           1       0.98      1.00      0.99        42

    accuracy                           0.99       146
   macro avg       0.99      1.00      0.99       146
weighted avg       0.99      0.99      0.99       146

AUC: 0.9990842490842491


# ****2DO MODELO (Incluyendo histopath_type, lesion_ISUP y lesion_GS)****
---
Este tiene la accuracy más alta, pero no representa algo real porque el modelo no aprende reglas (la solución se la da "lesion_GS")

DATA LOADING & EDA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
df = pd.read_csv('temp_database1.csv', sep=';')

In [ ]:
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

   patient_id  study_id    mri_date  patient_age   psa  psad  prostate_volume  \
0       11093   1001116  2020-02-06           56   4.9  0.20             24.0   
1       11345   1001368  2018-06-11           67   NaN   NaN             56.0   
2       10415   1000422  2020-06-23           65  15.0  0.27             55.0   
3       10406   1000413  2020-08-15           55   4.6  0.11             42.0   
4       10443   1000451  2019-03-17           64  17.0  0.22             78.0   

  histopath_type lesion_GS lesion_ISUP  case_ISUP case_csPCa  
0     SysBx+MRBx   3+4,3+3         2.1          2        YES  
1           MRBx       3+4           2          2        YES  
2            NaN       NaN         NaN          0         NO  
3     SysBx+MRBx   0+0,0+0           0          0         NO  
4          SysBx       0+0           0          0         NO  
(749, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 749 entries, 0 to 748
Data columns (total 12 columns):
 #   Column         

PREPROCESSING

In [ ]:
#Encoding of the target variable (if a patient has cancer or not: 1 if YES; O if NO)
df['case_csPCa'] = df['case_csPCa'].map({'YES': 1, 'NO': 0})
print(df.head())

   patient_id  study_id    mri_date  patient_age   psa  psad  prostate_volume  \
0       11093   1001116  2020-02-06           56   4.9  0.20             24.0   
1       11345   1001368  2018-06-11           67   NaN   NaN             56.0   
2       10415   1000422  2020-06-23           65  15.0  0.27             55.0   
3       10406   1000413  2020-08-15           55   4.6  0.11             42.0   
4       10443   1000451  2019-03-17           64  17.0  0.22             78.0   

  histopath_type lesion_GS lesion_ISUP  case_ISUP  case_csPCa  
0     SysBx+MRBx   3+4,3+3         2.1          2           1  
1           MRBx       3+4           2          2           1  
2            NaN       NaN         NaN          0           0  
3     SysBx+MRBx   0+0,0+0           0          0           0  
4          SysBx       0+0           0          0           0  


In [ ]:
#Drop columns that are not important (dates, type of analysis...)
cols_to_keep = ['patient_age', 'psa', 'psad', 'prostate_volume',
                'histopath_type', 'lesion_ISUP', 'lesion_GS', 'case_csPCa']
df = df[cols_to_keep]

#Null values before OHE
df['histopath_type'] = df['histopath_type'].fillna('Unknown')
df['lesion_ISUP'] = df['lesion_ISUP'].fillna('Missing')
df['lesion_GS'] = df['lesion_ISUP'].fillna('Missing')

#OHE
df = pd.get_dummies(df, columns=['histopath_type', 'lesion_ISUP', 'lesion_GS'])

#Adjusting the psa column
df = df[df['psa'] < 1000]

In [ ]:
print(df.describe())

       patient_age         psa        psad  prostate_volume  case_csPCa
count   727.000000  727.000000  524.000000       715.000000  727.000000
mean     65.895461   11.564127    0.216920        66.440448    0.290234
std       6.967067   14.243494    0.394606        36.836201    0.454183
min      37.000000    0.440000    0.010000        10.000000    0.000000
25%      61.500000    5.900000    0.090000        40.000000    0.000000
50%      66.000000    8.200000    0.140000        59.000000    0.000000
75%      71.000000   12.780000    0.220225        82.000000    1.000000
max      86.000000  210.000000    7.000000       279.000000    1.000000


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 727 entries, 0 to 748
Data columns (total 92 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_age                727 non-null    int64  
 1   psa                        727 non-null    float64
 2   psad                       524 non-null    float64
 3   prostate_volume            715 non-null    float64
 4   case_csPCa                 727 non-null    int64  
 5   histopath_type_MRBx        727 non-null    bool   
 6   histopath_type_RP          727 non-null    bool   
 7   histopath_type_SysBx       727 non-null    bool   
 8   histopath_type_SysBx+MRBx  727 non-null    bool   
 9   histopath_type_Unknown     727 non-null    bool   
 10  lesion_ISUP_0              727 non-null    bool   
 11  lesion_ISUP_0,0,0          727 non-null    bool   
 12  lesion_ISUP_0,0,1          727 non-null    bool   
 13  lesion_ISUP_0,2,2          727 non-null    bool   
 14 

In [ ]:
# Mean imputation for filling NAs (median imputation may be also applied if outliers)
df = df.fillna(df.median())

In [ ]:
print(df["case_csPCa"].value_counts())

case_csPCa
0    516
1    211
Name: count, dtype: int64


In [ ]:
#Target variable
X = df.drop(columns = ['case_csPCa'])
y = df['case_csPCa']

#Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [ ]:
#Normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

MLP MODEL

In [ ]:
def create_model(input_dim):
    model = models.Sequential([
        layers.Dense(32, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

model = create_model(X_train.shape[1])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.7050970873786407), np.int64(1): np.float64(1.7189349112426036)}


In [ ]:
#Training
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8211 - loss: 0.7343 - val_accuracy: 0.9060 - val_loss: 0.5677
Epoch 2/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9116 - loss: 0.4676 - val_accuracy: 0.8889 - val_loss: 0.4406
Epoch 3/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9547 - loss: 0.3478 - val_accuracy: 0.9573 - val_loss: 0.3284
Epoch 4/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9784 - loss: 0.2298 - val_accuracy: 0.9658 - val_loss: 0.2372
Epoch 5/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9914 - loss: 0.1564 - val_accuracy: 0.9658 - val_loss: 0.1669
Epoch 6/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9957 - loss: 0.1139 - val_accuracy: 0.9658 - val_loss: 0.1187
Epoch 7/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9978 - loss: 0.0733 - val_accuracy: 0.9744 - val_loss: 0.0899
Epoch 8/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9957 - loss: 0.0561 - val_accuracy: 0.9744 - val_loss

In [ ]:
#Evaluation
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.3).astype(int) #threshold of 0.3
print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       104
           1       0.98      1.00      0.99        42

    accuracy                           0.99       146
   macro avg       0.99      1.00      0.99       146
weighted avg       0.99      0.99      0.99       146

AUC: 0.9986263736263736


# ****3ER MODELO (sin incluir histopath_type, lesion_ISUP y lesion_GS)****
---
Quito estas variables porque puede que provoquen data leakage en los otros modelos y quiero ver qué resultados saldrían

DATA LOADING & EDA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
df = pd.read_csv('temp_database1.csv', sep=';')

In [ ]:
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

   patient_id  study_id    mri_date  patient_age   psa  psad  prostate_volume  \
0       11093   1001116  2020-02-06           56   4.9  0.20             24.0   
1       11345   1001368  2018-06-11           67   NaN   NaN             56.0   
2       10415   1000422  2020-06-23           65  15.0  0.27             55.0   
3       10406   1000413  2020-08-15           55   4.6  0.11             42.0   
4       10443   1000451  2019-03-17           64  17.0  0.22             78.0   

  histopath_type lesion_GS lesion_ISUP  case_ISUP case_csPCa  
0     SysBx+MRBx   3+4,3+3         2.1          2        YES  
1           MRBx       3+4           2          2        YES  
2            NaN       NaN         NaN          0         NO  
3     SysBx+MRBx   0+0,0+0           0          0         NO  
4          SysBx       0+0           0          0         NO  
(749, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 749 entries, 0 to 748
Data columns (total 12 columns):
 #   Column         

In [ ]:
print(df.describe())

         patient_id      study_id  patient_age           psa        psad  \
count    749.000000  7.490000e+02   749.000000  7.280000e+02  528.000000   
mean   10732.986649  1.000747e+06    65.925234  7.417582e+12    0.216451   
std      422.705211  4.305005e+02     6.952841  2.001373e+14    0.393311   
min    10002.000000  1.000002e+06    37.000000  4.400000e-01    0.010000   
25%    10359.000000  1.000365e+06    62.000000  5.900000e+00    0.090000   
50%    10721.000000  1.000737e+06    66.000000  8.210000e+00    0.135400   
75%    11095.000000  1.001118e+06    71.000000  1.286250e+01    0.220225   
max    11474.000000  1.001498e+06    86.000000  5.400000e+15    7.000000   

       prostate_volume   case_ISUP  
count       733.000000  749.000000  
mean         66.312306    0.950601  
std          36.712140    1.328006  
min          10.000000    0.000000  
25%          40.000000    0.000000  
50%          59.000000    0.000000  
75%          82.000000    2.000000  
max         279.000

In [ ]:
print(df['histopath_type'].unique())
print(df['lesion_GS'].unique())
print(df['lesion_ISUP'].unique())

['SysBx+MRBx' 'MRBx' nan 'SysBx' 'RP']
['3+4,3+3' '3+4' nan '0+0,0+0' '0+0' '4+4' 'N/A,3+4' '3+4,4+3' '3+3'
 '3+3,3+3' '3+3,N/A,N/A,N/A' '4+5,N/A' '4+3' '4+4,0+0,N/A' '3+4,3+4' '4+5'
 '5+4' '2+3,N/A,N/A' 'N/A,0+0,0+0' '3+3,0+0' '4+3,3+4' '0+0,4+4' '4+3,0+0'
 '4+4,N/A' '4+3,0+0,N/A,4+4,3+3' '3+3,N/A' '4+3,3+3' '3+4,4+4' '3+4,0+0'
 '3+4,N/A,0+0' '3+3,0+0,0+0' '0+0,3+4' '4+4,3+2' '3+4,N/A,3+3'
 '3+4,3+3,3+4,3+4' '3+2,N/A' '5+5,5+5' 'N/A,3+3,0+0' '0+0,N/A'
 '3+4,0+0,3+3' '3+3,3+3,0+0' 'N/A,0+0' '5+3' '3+5' '5+4,4+3,4+5'
 '3+4,3+4,0+0' '0+0,N/A,N/A' '4+3,3+3,3+3' '3+3,3+4' '5+5' 'N/A,3+3'
 'N/A,0+0,N/A' '0+0,3+3' '3+4,0+0,N/A' '3+5,3+3,3+3' '0+0,0+0,0+0'
 '0+0,0+0,N/A' '3+4,N/A' '4+4,4+3' '0+0,N/A,3+3' 'N/A,4+3' '0+0,3+4,3+4'
 '4+5,4+5' '0+0,N/A,0+0,0+0' '3+3,0+0,N/A' '3+3,3+3,3+3' '4+3,N/A'
 'N/A,N/A,0+0,0+0' '3+4,3+3,N/A' '4+5,3+3' '4+4,0+0' '3+4,4+3,3+3'
 '0+0,0+0,3+3' '0+0,4+3' '4+5,0+0,0+0,N/A' '3+4,0+0,3+4' '3+4,3+3,3+3'
 '2+3' '4+3,4+3']
['2.1' '2' nan '0' '4' '2.3' '1' '1.1' '5' '3'

In [ ]:
print(df['histopath_type'].value_counts())
print(df['lesion_GS'].value_counts())
print(df['lesion_ISUP'].value_counts())

histopath_type
MRBx          278
SysBx+MRBx    106
SysBx         103
RP             14
Name: count, dtype: int64
lesion_GS
0+0                122
3+4                 75
3+3                 59
0+0,0+0             32
4+3                 28
                  ... 
4+5,0+0,0+0,N/A      1
3+4,0+0,3+4          1
3+4,3+3,3+3          1
2+3                  1
4+3,4+3              1
Name: count, Length: 78, dtype: int64
lesion_ISUP
0          169
2           94
1           81
3           38
5           18
4           18
2.1         16
0.1         12
1.1         11
0,0,0        4
3.1          3
2.2          3
1.2          3
1,1,1        2
2,0,1        2
5.5          2
0.3          2
2.3          1
4.1          1
2.4          1
1,0,0        1
3,0,4,1      1
0.4          1
3.2          1
2,2,0        1
5,3,5        1
1,1,0        1
2,1,2,2      1
0.2          1
4,1,1        1
4.3          1
3,1,1        1
0,2,2        1
5.1          1
2,3,1        1
0,0,1        1
5,0,0        1
2,0,2        1
2,1,

PREPROCESSING

In [ ]:
#Encoding of the target variable (if a patient has cancer or not: 1 if YES; O if NO)
df['case_csPCa'] = df['case_csPCa'].map({'YES': 1, 'NO': 0})
print(df.head())

   patient_id  study_id    mri_date  patient_age   psa  psad  prostate_volume  \
0       11093   1001116  2020-02-06           56   4.9  0.20             24.0   
1       11345   1001368  2018-06-11           67   NaN   NaN             56.0   
2       10415   1000422  2020-06-23           65  15.0  0.27             55.0   
3       10406   1000413  2020-08-15           55   4.6  0.11             42.0   
4       10443   1000451  2019-03-17           64  17.0  0.22             78.0   

  histopath_type lesion_GS lesion_ISUP  case_ISUP  case_csPCa  
0     SysBx+MRBx   3+4,3+3         2.1          2           1  
1           MRBx       3+4           2          2           1  
2            NaN       NaN         NaN          0           0  
3     SysBx+MRBx   0+0,0+0           0          0           0  
4          SysBx       0+0           0          0           0  


In [ ]:
#Drop columns that are not important (dates, type of analysis...)
cols_to_keep = ['patient_age', 'psa', 'psad', 'prostate_volume','case_csPCa']
df = df[cols_to_keep]

#Adjusting the psa column
df = df[df['psa'] < 1000]

In [ ]:
print(df.describe())

       patient_age         psa        psad  prostate_volume  case_csPCa
count   727.000000  727.000000  524.000000       715.000000  727.000000
mean     65.895461   11.564127    0.216920        66.440448    0.290234
std       6.967067   14.243494    0.394606        36.836201    0.454183
min      37.000000    0.440000    0.010000        10.000000    0.000000
25%      61.500000    5.900000    0.090000        40.000000    0.000000
50%      66.000000    8.200000    0.140000        59.000000    0.000000
75%      71.000000   12.780000    0.220225        82.000000    1.000000
max      86.000000  210.000000    7.000000       279.000000    1.000000


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 727 entries, 0 to 748
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   patient_age      727 non-null    int64  
 1   psa              727 non-null    float64
 2   psad             524 non-null    float64
 3   prostate_volume  715 non-null    float64
 4   case_csPCa       727 non-null    int64  
dtypes: float64(3), int64(2)
memory usage: 34.1 KB
None


In [ ]:
# Mean imputation for filling NAs (median imputation may be also applied if outliers)
df = df.fillna(df.median())

In [ ]:
print(df["case_csPCa"].value_counts())

case_csPCa
0    516
1    211
Name: count, dtype: int64


In [ ]:
#Target variable
X = df.drop(columns = ['case_csPCa'])
y = df['case_csPCa']

#Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [ ]:
#Normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

MLP MODEL

In [ ]:
def create_model(input_dim):
    model = models.Sequential([
        layers.Dense(32, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

model = create_model(X_train.shape[1])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.7050970873786407), np.int64(1): np.float64(1.7189349112426036)}


In [ ]:
#Training
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.4634 - loss: 0.6919 - val_accuracy: 0.5726 - val_loss: 0.7256
Epoch 2/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5776 - loss: 0.6681 - val_accuracy: 0.6923 - val_loss: 0.7232
Epoch 3/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6422 - loss: 0.6459 - val_accuracy: 0.7265 - val_loss: 0.7101
Epoch 4/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6638 - loss: 0.6253 - val_accuracy: 0.7265 - val_loss: 0.7107
Epoch 5/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6724 - loss: 0.6217 - val_accuracy: 0.7265 - val_loss: 0.7024
Epoch 6/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6810 - loss: 0.6067 - val_accuracy: 0.7265 - val_loss: 0.7016
Epoch 7/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6810 - loss: 0.6063 - val_accuracy: 0.7350 - val_loss: 0.6970
Epoch 8/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6703 - loss: 0.6076 - val_accuracy: 0.7265 - val_loss

In [ ]:
#Evaluation
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.6).astype(int) #threshold of 0.6
print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
              precision    recall  f1-score   support

           0       0.85      0.81      0.83       104
           1       0.57      0.64      0.61        42

    accuracy                           0.76       146
   macro avg       0.71      0.73      0.72       146
weighted avg       0.77      0.76      0.76       146

AUC: 0.7744963369963371
